# Presidio Pipeline Checksum, Context, and Confidence Walkthrough

This notebook demonstrates the full Presidio analysis flow plus this repository's pipeline layer. It uses a small mock Vietnamese dataset so it can run without downloading external datasets.

The walkthrough covers custom recognizer registration, checksum validation, context score boosting, duplicate and conflict resolution, score thresholds, decision process output, and anonymization.

## Environment setup

The notebook uses `en_core_web_sm` as a local spaCy tokenizer/NLP model while registering recognizers under language code `vi`. This keeps the example deterministic and avoids downloading a Vietnamese model.

In [ ]:
import os
import sys
from pathlib import Path

# Keep third-party caches writable inside restricted notebook environments.
os.environ.setdefault("TLDEXTRACT_CACHE", "/tmp/tldextract-cache")

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "pipeline").exists():
            return candidate
    raise RuntimeError("Could not find project root containing src/pipeline")

project_root = find_project_root(Path.cwd().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display

from presidio_analyzer import (
    AnalyzerEngine,
    EntityRecognizer,
    Pattern,
    PatternRecognizer,
    RecognizerRegistry,
)
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine

from src.pipeline.BasePipeline import PIIPipeline
from src.pipeline.Evaluator import PIIEvaluator
from src.pipeline.Recognizers.CustomPatternRecognizer import CustomPatternRecognizer

In [ ]:
NLP_CONFIG = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "vi", "model_name": "en_core_web_sm"}],
}

nlp_engine = NlpEngineProvider(nlp_configuration=NLP_CONFIG).create_engine()

def make_analyzer(recognizers, log_decision_process: bool = False) -> AnalyzerEngine:
    registry = RecognizerRegistry(supported_languages=["vi"])
    for recognizer in recognizers:
        registry.add_recognizer(recognizer)
    if not registry.recognizers:
        raise ValueError("Pass at least one recognizer to avoid loading predefined recognizers")
    return AnalyzerEngine(
        registry=registry,
        nlp_engine=nlp_engine,
        supported_languages=["vi"],
        log_decision_process=log_decision_process,
    )

## Display helpers

The helpers below keep every section compact while still exposing spans, scores, recognizer metadata, validation status, and context explanations.

In [ ]:
def result_rows(text, results):
    rows = []
    for result in sorted(results, key=lambda item: (item.start, item.end, item.entity_type, -item.score)):
        explanation = getattr(result, "analysis_explanation", None)
        metadata = getattr(result, "recognition_metadata", {}) or {}
        rows.append(
            {
                "entity": result.entity_type,
                "span": text[result.start : result.end],
                "start": result.start,
                "end": result.end,
                "score": round(result.score, 3),
                "recognizer": metadata.get("recognizer_name", ""),
                "pattern": getattr(explanation, "pattern_name", None) if explanation else None,
                "original_score": getattr(explanation, "original_score", None) if explanation else None,
                "explanation_score": getattr(explanation, "score", None) if explanation else None,
                "score_context_improvement": getattr(explanation, "score_context_improvement", None) if explanation else None,
                "supportive_context": getattr(explanation, "supportive_context_word", None) if explanation else None,
                "validation": getattr(explanation, "validation_result", None) if explanation else None,
            }
        )
    return pd.DataFrame(rows)

def show_results(title, text, results):
    print(title)
    print(text)
    display(result_rows(text, results))

def raw_recognizer_results(text, recognizers, entities=None):
    nlp_artifacts = nlp_engine.process_text(text, "vi")
    raw_results = []
    requested_entities = entities or sorted({entity for recognizer in recognizers for entity in recognizer.supported_entities})
    for recognizer in recognizers:
        raw_results.extend(recognizer.analyze(text=text, entities=requested_entities, nlp_artifacts=nlp_artifacts))
    return raw_results

def mask_items(text, specs):
    items = []
    for value, label in specs:
        start = text.index(value)
        end = start + len(value)
        items.append({"start": start, "end": end, "label": label, "value": value})
    return items

## Mock Vietnamese dataset

The dataframe uses the `source_text` and `privacy_mask` fields expected by `PIIEvaluator`. The labels are the original dataset-style labels which can be mapped to Presidio entity names by the evaluator.

In [ ]:
mock_records = []

text_1 = "Khách hàng Nguyễn Văn A có email an.nguyen@example.vn, CCCD 123456786, điện thoại 0912345678."
mock_records.append(
    {
        "source_text": text_1,
        "privacy_mask": mask_items(
            text_1,
            [
                ("Nguyễn Văn A", "HO_VA_TEN"),
                ("an.nguyen@example.vn", "DIA_CHI_EMAIL"),
                ("123456786", "SO_CCCD_CMND"),
                ("0912345678", "SO_DIEN_THOAI"),
            ],
        ),
    }
)

text_2 = "Hồ sơ Trần Thị B có CCCD 123456789 và STK 9876543210 tại ngân hàng ABC."
mock_records.append(
    {
        "source_text": text_2,
        "privacy_mask": mask_items(
            text_2,
            [
                ("Trần Thị B", "HO_VA_TEN"),
                ("123456789", "SO_CCCD_CMND"),
                ("9876543210", "SO_TAI_KHOAN"),
                ("ABC", "TEN_NGAN_HANG"),
            ],
        ),
    }
)

text_3 = "Cần gửi biên nhận đến support.user@example.com và mã giao dịch MA-ABC123."
mock_records.append(
    {
        "source_text": text_3,
        "privacy_mask": mask_items(
            text_3,
            [
                ("support.user@example.com", "DIA_CHI_EMAIL"),
                ("MA-ABC123", "MA_GIAO_DICH"),
            ],
        ),
    }
)

mock_df = pd.DataFrame(mock_records)
display(mock_df[["source_text", "privacy_mask"]])

## Baseline Presidio analyzer

This baseline is a plain `AnalyzerEngine` with two simple notebook-local pattern recognizers. It is not using the repository pipeline yet.

In [ ]:
baseline_recognizers = [
    PatternRecognizer(
        supported_entity="ID",
        patterns=[Pattern("baseline_9_digit_id", r"\b\d{9}\b", 0.3)],
        supported_language="vi",
        name="baseline_id_regex",
    ),
    PatternRecognizer(
        supported_entity="EMAIL_ADDRESS",
        patterns=[Pattern("baseline_email", r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", 0.3)],
        supported_language="vi",
        name="baseline_email_regex",
    ),
]

baseline_analyzer = make_analyzer(baseline_recognizers, log_decision_process=True)
baseline_text = mock_df.loc[0, "source_text"]
baseline_results = baseline_analyzer.analyze(
    text=baseline_text,
    language="vi",
    return_decision_process=True,
)
show_results("Baseline AnalyzerEngine results", baseline_text, baseline_results)

## Repository pipeline with custom pattern recognizers

`PIIPipeline` registers the repository `CustomPatternRecognizer` objects into the underlying Presidio analyzer. The raw recognizer run shows the regex scores before analyzer-level context enhancement, duplicate removal, and score threshold filtering.

In [ ]:
repo_pattern_wrapper = CustomPatternRecognizer()
repo_pattern_wrapper.load_model()

raw_repo_results = raw_recognizer_results(baseline_text, repo_pattern_wrapper.recognizers)
show_results("Raw CustomPatternRecognizer recognizer results", baseline_text, raw_repo_results)

pipeline_base_analyzer = make_analyzer([baseline_recognizers[0]], log_decision_process=True)
pipeline = PIIPipeline(analyzer=pipeline_base_analyzer, recognizers=[CustomPatternRecognizer()])
pipeline.load_model()

pipeline_results = pipeline.analyzer.analyze(
    text=baseline_text,
    language="vi",
    return_decision_process=True,
)
show_results("PIIPipeline results after analyzer processing", baseline_text, pipeline_results)

## Checksum validation

`PatternRecognizer.validate_result()` can promote valid matches to `1.0` or remove invalid matches by setting their score to Presidio's minimum score.

The checksum rule here is deliberately small for demonstration: for a 9 digit ID, the last digit must equal the sum of the first eight digits modulo 10.

In [ ]:
class VietnameseChecksumIdRecognizer(PatternRecognizer):
    def __init__(self):
        super().__init__(
            supported_entity="ID",
            patterns=[Pattern("checksum_9_digit_id", r"\b\d{9}\b", 0.45)],
            supported_language="vi",
            context=["cccd", "cmnd", "căn", "cước"],
            name="vietnamese_checksum_id",
        )

    def validate_result(self, pattern_text: str):
        digits = [int(char) for char in pattern_text if char.isdigit()]
        if len(digits) != 9:
            return False
        return sum(digits[:8]) % 10 == digits[8]

checksum_analyzer = make_analyzer([VietnameseChecksumIdRecognizer()], log_decision_process=True)
checksum_text = "CCCD hợp lệ 123456786 và CCCD sai 123456789."
checksum_results = checksum_analyzer.analyze(
    text=checksum_text,
    language="vi",
    return_decision_process=True,
)
show_results("Checksum recognizer results", checksum_text, checksum_results)

assert any(checksum_text[item.start : item.end] == "123456786" and item.score == 1.0 for item in checksum_results)
assert all(checksum_text[item.start : item.end] != "123456789" for item in checksum_results)

## Register checksum recognizer into the pipeline analyzer

The checksum recognizer is notebook-local, but it is still a Presidio recognizer and can be registered into the analyzer that `PIIPipeline` owns.

In [ ]:
pipeline.analyzer.registry.add_recognizer(VietnameseChecksumIdRecognizer())

pipeline_checksum_results = pipeline.analyzer.analyze(
    text=baseline_text,
    language="vi",
    return_decision_process=True,
)
show_results("Pipeline results with checksum recognizer registered", baseline_text, pipeline_checksum_results)

## Context score boosting

The default enhancer is `LemmaContextAwareEnhancer`. With the local installed Presidio version, it adds `0.35`, applies a minimum score of `0.4` when context is found, and caps the final score at `1.0`.

In [ ]:
enhancer = pipeline.analyzer.context_aware_enhancer
display(
    pd.DataFrame(
        [
            {
                "enhancer": enhancer.__class__.__name__,
                "context_similarity_factor": enhancer.context_similarity_factor,
                "min_score_with_context_similarity": enhancer.min_score_with_context_similarity,
                "context_prefix_count": enhancer.context_prefix_count,
                "context_suffix_count": enhancer.context_suffix_count,
            }
        ]
    )
)

In [ ]:
weak_email = PatternRecognizer(
    supported_entity="EMAIL_ADDRESS",
    patterns=[Pattern("weak_email", r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", 0.2)],
    supported_language="vi",
    context=["email"],
    name="weak_email_with_context",
)
weak_bank = PatternRecognizer(
    supported_entity="BANK_ACCOUNT",
    patterns=[Pattern("weak_bank_account", r"\b\d{8,15}\b", 0.2)],
    supported_language="vi",
    context=["stk", "ngân", "khoản"],
    name="weak_bank_with_context",
)
context_analyzer = make_analyzer([weak_email, weak_bank], log_decision_process=True)

email_without_context = "Gửi đến weak.user@example.com ngay hôm nay."
email_with_context = "Email liên hệ weak.user@example.com ngay hôm nay."
bank_without_context = "Lưu lại 9876543210 để đối chiếu."
bank_with_context = "STK 9876543210 tại ngân hàng ACB."

for title, text, kwargs in [
    ("Email without text context", email_without_context, {}),
    ("Email with nearby text context", email_with_context, {}),
    ("Email with analyzer-level external context", email_without_context, {"context": ["email"]}),
    ("Bank account without text context", bank_without_context, {}),
    ("Bank account with nearby text context", bank_with_context, {}),
]:
    results = context_analyzer.analyze(
        text=text,
        language="vi",
        return_decision_process=True,
        **kwargs,
    )
    show_results(title, text, results)

## Duplicate and conflict filtering

Presidio runs `EntityRecognizer.remove_duplicates()` after context enhancement. The local implementation sorts by score descending, then start offset, then longer span. Same-type results contained in an already kept higher-ranked result are removed.

In [ ]:
conflict_text = "Mã giao dịch MA-ABC123 đã hoàn tất."
long_high = PatternRecognizer(
    supported_entity="ID",
    patterns=[Pattern("full_transaction_id", r"\bMA-ABC123\b", 0.8)],
    supported_language="vi",
    name="long_high_transaction_id",
)
short_low = PatternRecognizer(
    supported_entity="ID",
    patterns=[Pattern("inner_transaction_id", r"\bABC123\b", 0.4)],
    supported_language="vi",
    name="short_low_transaction_id",
)
conflict_recognizers = [long_high, short_low]
raw_conflicts = raw_recognizer_results(conflict_text, conflict_recognizers, entities=["ID"])
deduped_conflicts = EntityRecognizer.remove_duplicates(raw_conflicts)
conflict_analyzer = make_analyzer(conflict_recognizers, log_decision_process=True)
analyzer_conflicts = conflict_analyzer.analyze(
    text=conflict_text,
    language="vi",
    return_decision_process=True,
)

show_results("Raw overlapping recognizer results", conflict_text, raw_conflicts)
show_results("Direct remove_duplicates output", conflict_text, deduped_conflicts)
show_results("Analyzer output after duplicate removal", conflict_text, analyzer_conflicts)

## Score threshold filtering

The analyzer removes duplicates before applying `score_threshold`. The weak result below is present with the default threshold and filtered out when the threshold is raised.

In [ ]:
threshold_text = "Mã tạm thời 246813579 cần kiểm tra lại."
low_score_id = PatternRecognizer(
    supported_entity="ID",
    patterns=[Pattern("low_score_9_digit_id", r"\b\d{9}\b", 0.25)],
    supported_language="vi",
    name="low_score_id",
)
threshold_analyzer = make_analyzer([low_score_id], log_decision_process=True)

threshold_default = threshold_analyzer.analyze(
    text=threshold_text,
    language="vi",
    return_decision_process=True,
)
threshold_filtered = threshold_analyzer.analyze(
    text=threshold_text,
    language="vi",
    score_threshold=0.5,
    return_decision_process=True,
)

show_results("Default score threshold", threshold_text, threshold_default)
show_results("score_threshold=0.5", threshold_text, threshold_filtered)
print({"default_count": len(threshold_default), "threshold_0_5_count": len(threshold_filtered)})

## Evaluator-compatible pipeline call

This quick evaluator run is not intended to optimize metrics. It verifies that the mock dataframe shape works with the repository evaluator and pipeline wrapper.

In [ ]:
evaluator = PIIEvaluator()
evaluation = evaluator.evaluate_presidio(
    mock_df,
    pipeline,
    language="vi",
    use_type_mapping=True,
)
display(pd.DataFrame([evaluation]))

## Anonymization

`AnonymizerEngine` consumes the final Presidio analyzer results. The example filters to entity types that do not overlap in this text so the anonymized output is easy to inspect.

In [ ]:
anonymizer = AnonymizerEngine()
final_text = baseline_text
final_results = pipeline.analyzer.analyze(
    text=final_text,
    language="vi",
    entities=["EMAIL_ADDRESS", "ID", "PHONE_NUMBER"],
    return_decision_process=True,
)
show_results("Final analyzer results for anonymization", final_text, final_results)

anonymized = anonymizer.anonymize(text=final_text, analyzer_results=final_results)
print(anonymized.text)

## JSONL audit logging

`PIIPipeline` can append one audit record per input to a shared JSONL file. Each derived pipeline writes the same `run_id` and its own `pipeline_name`, so comparison can happen by filtering or grouping the loaded dataframe. Raw source text remains disabled by default.


In [ ]:
audit_log_path = project_root / "output" / "predictions.jsonl"
if audit_log_path.exists():
    audit_log_path.unlink()

audit_texts = mock_df["source_text"].head(2).tolist()
audit_input_ids = [f"row-{idx:03d}" for idx in range(len(audit_texts))]
audit_ground_truth = mock_df["privacy_mask"].head(2).tolist()
audit_run_id = "2026-06-09T04:20:00Z"

audit_pipeline_specs = [
    ("baseline_presidio", make_analyzer(baseline_recognizers, log_decision_process=True), None),
    ("regex_only", make_analyzer([baseline_recognizers[0]], log_decision_process=True), [CustomPatternRecognizer()]),
    ("full_hybrid_pipeline", make_analyzer([baseline_recognizers[0]], log_decision_process=True), [CustomPatternRecognizer()]),
]

for pipeline_name, analyzer, extra_recognizers in audit_pipeline_specs:
    audit_pipeline = PIIPipeline(
        analyzer=analyzer,
        recognizers=extra_recognizers,
        pipeline_name=pipeline_name,
        run_id=audit_run_id,
        prediction_log_path=audit_log_path,
    )
    audit_pipeline.load_model()
    audit_pipeline.predict(
        audit_texts,
        language="vi",
        score_threshold=0.4,
        input_ids=audit_input_ids,
        ground_truth=audit_ground_truth,
    )

audit_df = pd.read_json(audit_log_path, lines=True)
display(audit_df[["run_id", "pipeline_name", "input_id", "language", "score_threshold", "source_text"]])
display(audit_df.groupby("pipeline_name").size().rename("record_count").reset_index())


## What this notebook demonstrated

1. A plain Presidio `AnalyzerEngine` run with simple regex recognizers.
2. Repository pipeline registration through `PIIPipeline` and `CustomPatternRecognizer`.
3. Checksum validation promoting a valid match to `1.0` and dropping an invalid match.
4. Context boosting from nearby text and analyzer-level external context.
5. Duplicate removal before score threshold filtering.
6. Decision process fields when `return_decision_process=True`.
7. Anonymization using final Presidio results.
8. Shared JSONL audit logging across named pipeline variants.
